In [ ]:
import requests
import pandas as pd
import re
from typing import Dict, Any, List

In [ ]:
def get_uniprot(id_value: str):
    '''
    Get data from Uniprot API by ID - returns requests.Response object
    '''
    url = f"https://rest.uniprot.org/uniprotkb/{id_value}.json"
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    return response


In [ ]:
get_uniprot('helloworld').json()

{'url': 'http://rest.uniprot.org/uniprotkb/helloworld',
 'messages': ["The 'accession' value has invalid format. It should be a valid UniProtKB accession"]}

In [ ]:
def uniprot_parse_response(resp):
    '''
    Parse Uniprot response and output: organism, geneInfo, sequenceInfo, type
    '''
    try:
        data = resp.json()
        organism = data.get("organism", {}).get("scientificName", "NA")
        genes = data.get("genes", [])
        gene_info = genes[0] if genes else {}
        sequence_info = data.get("sequence", {})
        entry_type = data.get("uniProtkbEntryType", "protein")

        return {
            "organism": organism,
            "geneInfo": gene_info,
            "sequenceInfo": sequence_info,
            "type": entry_type
        }
    except:
        return {"error": "Parse failed"}

In [ ]:
uniprot_parse_response(get_uniprot('P11473'))

{'organism': 'Homo sapiens',
 'geneInfo': {'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
     'source': 'HGNC',
     'id': 'HGNC:12679'}],
   'value': 'VDR'},
  'synonyms': [{'value': 'NR1I1'}]},
 'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
  'length': 427,
  'molWeight': 48289,
  'crc64': 'F95F300D042C4CB7',
  'md5': '0D963ACD4A34674368324EE026023597'},
 'type': 'protein'}

In [ ]:
def get_ensembl(id_value: str):
    '''
    Get data from Ensembl API by ID - returns requests.Response object
    '''
    server = "https://rest.ensembl.org"
    ext = f"lookup/id/{id_value}?"

    http_args = {
        "method": "GET",
        "headers": {"Accept": "application/json"}
    }

    response = requests.request(**http_args, url=f"{server}/{ext}")
    return response

In [ ]:
get_ensembl('helloworld').json()

{'error': "ID 'helloworld' not found"}

In [ ]:
def ensembl_parse_response(resp):
    '''
    Parse Ensembl response and output: object_type, assembly_name, species, db_type, biotype, display_name, id, description, canonical_transcript, source
    '''
    try:
        data = resp.json()
        if "error" in data:
            return {"error": data["error"]}

        return {
            "object_type": data.get("object_type", "NA"),
            "species": data.get("species", "NA"),
            "assembly_name": data.get("assembly_name", "NA"),
            "biotype": data.get("biotype", "NA"),
            "display_name": data.get("display_name", "NA"),
            "id": data.get("id", "NA"),
            "db_type": data.get("db_type", "NA"),
            "description": data.get("description", "NA"),
            "source": data.get("source", "NA"),
            "canonical_transcript": data.get("canonical_transcript", "NA")
        }
    except:
        return {"error": "Parse failed"}

In [ ]:
ensembl_parse_response(get_ensembl('ENSMUSG00000041147'))

{'object_type': 'Gene',
 'species': 'mus_musculus',
 'assembly_name': 'GRCm39',
 'biotype': 'protein_coding',
 'display_name': 'Brca2',
 'id': 'ENSMUSG00000041147',
 'db_type': 'core',
 'description': 'breast cancer 2, early onset [Source:MGI Symbol;Acc:MGI:109337]',
 'source': 'ensembl_havana',
 'canonical_transcript': 'ENSMUST00000044620.11'}

In [ ]:
def main(ids: List[str]) -> Dict[str, Dict]:
    '''
    Main function that iterates over IDs, determines database by regex,
    calls appropriate get/parse functions, returns dict {ID: parsed_data}
    '''
    results = {}

    # Regex patterns from test examples
    uniprot_pattern = re.compile(r'^[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9][A-Z][A-Z0-9]{2}[0-9]')
    ensembl_pattern = re.compile(r'^ENS[A-Z]*G\d+$|^ENST\d+$|^ENSMUSG\d+$')

    for id_value in ids:
        if uniprot_pattern.match(id_value):
            resp = get_uniprot(id_value)
            results[id_value] = uniprot_parse_response(resp)
        elif ensembl_pattern.match(id_value):
            resp = get_ensembl(id_value)
            results[id_value] = ensembl_parse_response(resp)
        else:
            results[id_value] = {"error": "unknown database"}

    return results

In [ ]:
main(['P11473', 'Q91XI3', 'hello', 'ENSG00000157764', 'ENSG00000139618'])

{'P11473': {'organism': 'Homo sapiens',
  'geneInfo': {'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312',
      'source': 'HGNC',
      'id': 'HGNC:12679'}],
    'value': 'VDR'},
   'synonyms': [{'value': 'NR1I1'}]},
  'sequenceInfo': {'value': 'MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS',
   'length': 427,
   'molWeight': 48289,
   'crc64': 'F95F300D042C4CB7',
   'md5': '0D963ACD4A34674368324EE026023597'},
  'type': 'protein'},
 'Q91XI3': {'organism': 'Ictidomys tridecemlineatus',
  'geneInfo': {'geneName': {'value': 'INS'}},
  'sequenceInfo': {'value': 'MALWTRLLPLLALLALLGPDPAQAFVNQHLCGSHLVEALYLVCGE